# Exploring a neutrino-factory HDF5 output file

This notebook is a guided tour of the **common output format**: the single HDF5
layout every generator (GENIE, NuWro, NEUT, GiBUU) is normalized onto. It is
meant to be read top to bottom the first time — each section explains one part
of the file before plotting it.

By the end you will have:

1. looked inside the file with `h5py` and seen every group, attribute and column;
2. plotted the event breakdown by interaction type and the energy spectrum,
   raw and weighted;
3. plotted the **cross section vs. energy** per interaction type, using
   `neutrino_factory.plots`;
4. histogrammed **every kinematic variable weighted by `xsec_weight`**;
5. picked an **energy slice** and plotted the double differential cross section
   d²σ/dx dy inside it.

## Before you start

Jupyter is an *optional* dependency of the project — the CLI and the plotting
module work without it. Install the notebook extras once:

```bash
pip install -e ".[notebook]"
jupyter lab
```

You also need at least one merged output file. If you have none, a synthetic
one is two commands away (no generator containers required):

```bash
neutrino-factory submit --config configs/examples/power_law_numu_Ar.yaml --executor local
```

That runs in `stub_mode`, so its kinematic columns are all placeholders — good
enough to exercise sections 1–6, but sections 7 and 8 will be empty. For real
kinematics use one of `configs/smoke/*.yaml` with the corresponding container
image built.

Related reading: `docs/physics.md` (the physics contract), the
`## Common output format` section of the repository `README.md`, and
`docs/architecture.md` (where outputs land).

## 0. Setup

We import the *axis-level* helpers from `neutrino_factory.plots` — the functions
that draw onto an axes object you own (`plot_interactions`, `plot_energy`,
`plot_xsec_by_interaction`) plus the figure-level `figure_xsec`, which takes
`plt` as an argument.

We deliberately do **not** call `make_plots` / `make_config_plots` here: those
are the batch entry points, and they switch matplotlib to the non-interactive
`Agg` backend and write PNG files. Inside a notebook you want the inline
backend and figures on screen. (Use the CLI —
`neutrino-factory plot-output --config ...` — when you want the PNGs.)

In [ ]:
%matplotlib inline

import os
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

from neutrino_factory import kinematics
from neutrino_factory.common_output import EVENT_NUMERIC_FIELDS, EVENT_STRING_FIELDS
from neutrino_factory.plots import (
    dataset_title,
    figure_xsec,
    interaction_counts,
    log_bin_edges,
    plot_energy,
    plot_interactions,
    plot_xsec_by_interaction,
    read_plot_data,
)

plt.rcParams["figure.dpi"] = 110

### Choosing a file

Set `DATA_FILE` to the merged HDF5 file you want to look at. The cell below
falls back to discovering `output/merged/*.h5` under the repository root, and
prints everything it found so you can point at a different one.

Merged files (`output/merged/`) are the ones to plot: a run splits into chunks,
and chunk files each carry a *per-chunk* estimate of the cross section, so
plotting one chunk of many understates nothing but plotting a naive
concatenation would overcount. Merging is what turns those estimates into an
average (see `docs/physics.md`, "Merging chunks averages, it does not sum").

In [ ]:
def repository_root(start=None) -> Path:
    """Walk up from `start` (default: the cwd) to the directory holding pyproject.toml."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


ROOT = repository_root()
MERGED_DIR = Path(os.environ.get("NF_OUTPUT_ROOT", ROOT / "output")) / "merged"

available = sorted(MERGED_DIR.glob("*.h5"))
print(f"merged outputs under {MERGED_DIR}:")
for path in available:
    print(f"  {path.name}")

# Point this at any file above (or an absolute path of your own).
DATA_FILE = available[0] if available else None
if DATA_FILE is None:
    raise SystemExit(
        f"No merged HDF5 files under {MERGED_DIR}. Run a job first — see the "
        "intro cell — or set DATA_FILE by hand."
    )
print(f"\nusing: {DATA_FILE}")

## 1. What is actually inside the file

Three top-level groups:

| group | what it holds |
|---|---|
| `metadata` | attributes only: generator identity (`generator`, `code_version`, `config_version`), the flux that was simulated, `expected_events`, the merge provenance |
| `run` | attributes only: `event_count`, the number of events actually written |
| `events` | one **1-D dataset per column**, all the same length — this is the event table |

Note the storage choice: `events` is *column-major* (one dataset per variable),
not an array of records. That is why loading only the columns you need is cheap,
and why every plotting helper below takes plain numpy arrays.

The cell below walks the file and prints the structure without assuming
anything about it, so it also works on files written by a future version that
added a column.

In [ ]:
def describe(path) -> None:
    """Print every group, attribute and dataset of a common-output file."""
    with h5py.File(path, "r") as handle:
        for group_name in handle:
            group = handle[group_name]
            print(f"/{group_name}")
            for key, value in group.attrs.items():
                text = value.decode() if isinstance(value, bytes) else value
                text = str(text)
                if len(text) > 90:
                    text = text[:87] + "..."
                print(f"    @{key} = {text}")
            for name in group:
                dataset = group[name]
                print(f"    {name:<26} shape={dataset.shape} dtype={dataset.dtype}")


describe(DATA_FILE)

The column names are not hard-coded in this notebook — the package declares
them, and everything downstream (the writer, the reader, the merger) is driven
off that one table. Worth printing once so you know what you can ask for:

In [ ]:
print("numeric columns:")
for name in EVENT_NUMERIC_FIELDS:
    print(f"  {name}")
print("\nstring columns:")
for name in EVENT_STRING_FIELDS:
    print(f"  {name}")
print("\nof which derived kinematics:", ", ".join(kinematics.KINEMATIC_FIELDS))

## 2. Loading the columns

Two ways in, and it is worth knowing both:

* **`read_plot_data(path)`** from `neutrino_factory.plots` returns a `PlotData`
  with exactly what the shipped plots need — energies, weights, cross-section
  weights, interaction labels, the CC/NC mask, and the metadata. Use it whenever
  you call one of the plotting helpers.
* **reading the columns yourself** with `h5py` gives you the kinematic columns
  too, which `PlotData` does not carry. We do both: `data` for the plot helpers,
  `cols` for the kinematics in sections 7 and 8.

(There is a third: `common_output.read_events` returns one dict *per event*.
Convenient for a handful of events, wasteful for a million — the plots want
arrays anyway.)

In [ ]:
def read_columns(path) -> tuple[dict, dict[str, np.ndarray]]:
    """Read the metadata attributes and every `events` column as a numpy array."""
    with h5py.File(path, "r") as handle:
        metadata = {
            key: (value.decode() if isinstance(value, bytes) else value)
            for key, value in handle["metadata"].attrs.items()
        }
        columns: dict[str, np.ndarray] = {}
        for name, dataset in handle["events"].items():
            values = dataset[()]
            if values.dtype.kind in ("S", "O", "U"):
                columns[name] = np.array(
                    [v.decode() if isinstance(v, bytes) else str(v) for v in values],
                    dtype=object,
                )
            else:
                columns[name] = np.asarray(values)
    return metadata, columns


data = read_plot_data(DATA_FILE)       # what the plots in neutrino_factory.plots want
metadata, cols = read_columns(DATA_FILE)  # every column, including the kinematics

print(dataset_title(data))
print(f"{len(data.energies):,} events written, {metadata['expected_events']:,} requested")
print(f"flux: {metadata['flux']}")

## 3. Placeholders: the columns that are not always defined

The framework refuses to invent physical values. Where a kinematic variable is
undefined for an event, the column carries a deliberately unphysical
placeholder rather than a plausible number:

* **`-1`** for the non-negative quantities (`q2_gev2`, `bjorken_x`,
  `inelasticity_y`, the lepton energies/momenta);
* **`-999`** for `lepton_p_parallel_gev` and `lepton_costheta`, whose physical
  range includes `-1`.

So `bjorken_x == -1` is not "x is minus one", it is "there is no struck nucleon
to define x against" — coherent scattering, for instance. **Always mask
placeholders out before histogramming**, or they pile up in a spike at the
placeholder value and quietly bias every mean you compute.

The per-field placeholder is `kinematics.FIELD_DEFAULTS`, which is also what
`common_output` builds its column table from — so the masking below stays
correct if a column is ever added.

In [ ]:
def defined(field: str) -> np.ndarray:
    """Boolean mask of events for which `field` holds a real value."""
    values = cols[field]
    mask = np.isfinite(values)
    placeholder = kinematics.FIELD_DEFAULTS.get(field)
    if placeholder is not None:
        mask &= values != placeholder
    return mask


total = len(cols["energy_gev"])
print(f"{'column':<26} {'defined':>10} {'placeholder':>12}")
for field in kinematics.KINEMATIC_FIELDS:
    n_defined = int(defined(field).sum())
    print(f"{field:<26} {n_defined:>10,} {total - n_defined:>12,}")

If the whole table above reads *0 defined*, you are looking at a `stub_mode`
file: the synthetic generator draws energies from the flux but produces no
outgoing lepton, so every derived kinematic column is a placeholder. Sections
4–6 still work; sections 7 and 8 need a real generator run.

## 4. What kind of events are in here?

`interaction` is the harmonized channel label — `qel`, `res`, `dis`, `coh`,
`mec`, `other` — mapped from each generator's own scheme by its normalizer.
`interaction_counts` puts the known labels in canonical order and appends
anything unexpected rather than dropping it (a `stub_mode` file shows a single
`inclusive` label this way).

The dashed line is the requested event count. A gap between it and the bars is
worth understanding before you trust anything else: generators can write fewer
events than asked.

In [ ]:
counts = interaction_counts(data.interactions)
fig, ax = plt.subplots(figsize=(9, 3))
plot_interactions(counts, int(data.metadata["expected_events"]), ax)
ax.set_title(dataset_title(data))
plt.show()

for label, count in counts.items():
    print(f"{label:<12} {count:>9,}  ({100 * count / max(len(data.interactions), 1):5.1f} %)")

## 5. The energy spectrum — raw and weighted

Two different questions:

* **Raw counts** answer *where did the generator spend its events?* This is the
  sampling distribution, and it is what determines your statistical
  uncertainties.
* **`weight`-weighted counts** apply each generator's own native weight column,
  passed through verbatim. For a rejection-sampling generator every weight is 1
  and the two panels are the same shape.

Neither of these is a cross section — that is section 6. Both histograms use
log-spaced energy bins and divide each bin by its width, so the y-axis is a
count *per GeV* and bins of unequal width stay comparable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
plot_energy(data.energies, axes[0], bins=60)
axes[0].set_title("raw (sampling distribution)")
plot_energy(data.energies, axes[1], weights=data.weights, bins=60)
axes[1].set_title("weighted by the native `weight` column")
fig.suptitle(dataset_title(data))
fig.tight_layout()
plt.show()

## 6. The cross section vs. energy

This is the plot the whole framework exists to produce. Its contract, stated
once on `ConfigTranslator.compute_xsec_weight` and honoured by every generator:

> Histogram events by energy, weight by `xsec_weight`, divide by the bin width →
> the average differential cross section in that bin, in 1e-38 cm² **per target
> nucleon**.

Per *nucleon*, always — cross sections on different nuclei are otherwise not
comparable quantities. By convention the plots show **σ(E)/E**, so the module
divides each bin by its geometric centre as well; that flattens the roughly
linear rise of the CC total cross section and makes deviations visible.

`plot_xsec_by_interaction` draws one line per channel plus a dashed total.
`figure_xsec` wraps it into one panel per weak current actually present in the
file — so an inclusive run gets a CC and an NC panel sharing a y-axis, and a
CC-only run stays a single panel.

In [ ]:
fig = figure_xsec(data, plt, bins=60)
fig.tight_layout()
plt.show()

Because it takes an axes, `plot_xsec_by_interaction` is also the building block
for your own selections. Here, the same quantity for one interaction channel
only — change `CHANNEL` (or the mask) to slice the sample however you like:

In [ ]:
CHANNEL = "qel"  # try: res, dis, coh, mec, other — or whatever section 4 listed

mask = data.interactions == CHANNEL
if mask.any():
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_xsec_by_interaction(
        data.energies[mask],
        data.xsec_weights[mask],
        data.interactions[mask],
        ax,
        # Share the binning of the whole file so this panel is directly
        # comparable with the one above.
        bin_edges=log_bin_edges(data.energies, 60),
        legend_title=CHANNEL.upper(),
    )
    ax.set_title(dataset_title(data))
    plt.show()
else:
    print(f"no {CHANNEL!r} events in this file; section 4 lists what is available")

## 7. Every kinematic variable, weighted by the cross section

**Weighting by `xsec_weight` here is a correctness requirement, not a
refinement.** GiBUU does not rejection-sample: it samples phase space uniformly
and weights each event by its cross section, so its *raw* event distribution is
not the physical one and an unweighted histogram of it is simply wrong. GENIE,
NuWro and NEUT do rejection-sample, so for them the weighted and raw shapes
agree — which is exactly why the weighted version is the one to standardize on.

Each panel below sums `xsec_weight` per bin and divides by the bin width, so it
reads as a differential cross section dσ/dv in 1e-38 cm² per nucleon per unit
of *v*. The shaded band is the weighted-count error √(Σw²) per bin, scaled the
same way — on a heavily weighted sample it can be startlingly wide, and that is
the honest answer.

Placeholders are masked out per variable using section 3's `defined()`.

In [ ]:
# (column, axis label). energy_gev leads: it is the one kinematic column with no
# placeholder, and it anchors the others.
PLOT_VARIABLES = (
    ("energy_gev", r"$E_\nu$ (GeV)"),
    ("q2_gev2", r"$Q^2$ (GeV$^2$)"),
    ("bjorken_x", r"Bjorken $x$"),
    ("inelasticity_y", r"inelasticity $y$"),
    ("lepton_energy_gev", r"$E_\ell$ (GeV)"),
    ("lepton_momentum_gev", r"$p_\ell$ (GeV)"),
    ("lepton_p_parallel_gev", r"$p_{\ell,\parallel}$ (GeV)"),
    ("lepton_p_transverse_gev", r"$p_{\ell,T}$ (GeV)"),
    ("lepton_costheta", r"$\cos\theta_\ell$"),
)


def xsec_density(values, weights, bin_edges):
    """Sum of weights per bin / bin width, with the weighted-count error."""
    sums, edges = np.histogram(values, bins=bin_edges, weights=weights)
    sum_sq, _ = np.histogram(values, bins=edges, weights=weights**2)
    widths = np.diff(edges)
    return sums / widths, np.sqrt(sum_sq) / widths, edges


def plot_variable(field, label, ax, bins=40):
    """One kinematic variable, weighted by xsec_weight, on `ax`."""
    mask = defined(field)
    values, weights = cols[field][mask], cols["xsec_weight"][mask]
    if values.size == 0:
        ax.text(0.5, 0.5, "no defined values", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(label)
        return

    edges = np.histogram_bin_edges(values, bins=bins)
    density, errors, edges = xsec_density(values, weights, edges)
    ax.stairs(density, edges, fill=False, color="C0")
    ax.stairs(
        density + errors, edges, baseline=density - errors, fill=True, color="C0", alpha=0.3
    )
    ax.set_xlabel(label)
    ax.set_ylabel(r"$d\sigma/dv$ ($10^{-38}\,\mathrm{cm}^2/\mathrm{nucleon}$)", fontsize="small")


fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, (field, label) in zip(np.ravel(axes), PLOT_VARIABLES):
    plot_variable(field, label, ax)
fig.suptitle(dataset_title(data))
fig.tight_layout()
plt.show()

A few things to look for, since these plots are also the fastest sanity check
on a normalization you do not yet trust:

* **`bjorken_x` peaks near 1 for quasi-elastic** scattering — x is defined
  against a fixed nucleon mass, so QE lands there by construction. A QE-dominated
  sample with no peak at 1 means something upstream is wrong.
* **`lepton_costheta` piles up near +1** at these energies: the outgoing lepton
  is forward.
* **`inelasticity_y` can be slightly negative** and that is physical, not a bug —
  Fermi motion of the struck nucleon can push the lepton energy just above the
  neutrino energy.
* Split any of these by interaction type by masking `cols["interaction"]`
  before calling `plot_variable`'s internals — the channels look very different,
  and averaging them together hides that.

## 8. An energy slice: the double differential cross section in x and y

Cross sections are usually quoted differentially in the two dimensionless
variables x and y at a fixed energy. Our samples are spread over a whole flux,
so the closest equivalent is a **narrow energy slice**: select events in
[E_lo, E_hi), histogram them in (x, y), sum `xsec_weight` per cell, and divide
by the cell area **and by the width of the slice**.

Why divide by the slice width too? Because the sum of `xsec_weight` over an
energy interval estimates ∫σ(E) dE over that interval — the flux density was
already divided out when the weight was built. Dividing by ΔE therefore turns
the sum back into an average over the slice, and the result reads as

> ⟨d²σ/dx dy⟩ over [E_lo, E_hi), in 1e-38 cm² per nucleon.

Make the slice too narrow and you run out of events; too wide and you are
averaging over genuinely different physics. Start where the flux has the most
events, and watch the event count the cell prints.

In [ ]:
# ---- choose your slice -------------------------------------------------------
E_LO_GEV, E_HI_GEV = 1.0, 2.0
X_BINS = np.linspace(0.0, 1.5, 31)
Y_BINS = np.linspace(0.0, 1.0, 31)
# ------------------------------------------------------------------------------


def double_differential(e_lo, e_hi, x_bins=X_BINS, y_bins=Y_BINS):
    """<d^2 sigma / dx dy> averaged over the energy slice [e_lo, e_hi).

    Returns the density array (x along the first axis), the bin edges, and how
    many events entered it.
    """
    if e_hi <= e_lo:
        raise ValueError(f"empty energy slice: [{e_lo}, {e_hi})")

    mask = (
        (cols["energy_gev"] >= e_lo)
        & (cols["energy_gev"] < e_hi)
        & defined("bjorken_x")
        & defined("inelasticity_y")
    )
    sums, x_edges, y_edges = np.histogram2d(
        cols["bjorken_x"][mask],
        cols["inelasticity_y"][mask],
        bins=[x_bins, y_bins],
        weights=cols["xsec_weight"][mask],
    )
    cell_area = np.outer(np.diff(x_edges), np.diff(y_edges))
    return sums / (cell_area * (e_hi - e_lo)), x_edges, y_edges, int(mask.sum())


def plot_double_differential(e_lo, e_hi, ax=None):
    density, x_edges, y_edges, n_events = double_differential(e_lo, e_hi)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5.5))

    # Cells with no events are empty, not zero cross section — leave them blank
    # rather than painting them the bottom colour of the scale.
    shown = np.ma.masked_where(density == 0.0, density)
    # `viridis` is perceptually uniform, so equal colour steps are equal steps in
    # the cross section; the transpose puts x on the horizontal axis.
    mesh = ax.pcolormesh(x_edges, y_edges, shown.T, cmap="viridis", shading="flat")
    bar = ax.figure.colorbar(mesh, ax=ax)
    bar.set_label(
        r"$\langle d^2\sigma/dx\,dy\rangle$ "
        r"($10^{-38}\,\mathrm{cm}^2/\mathrm{nucleon}$)"
    )
    ax.set_xlabel(r"Bjorken $x$")
    ax.set_ylabel(r"inelasticity $y$")
    ax.set_title(f"{e_lo:g} – {e_hi:g} GeV   ({n_events:,} events)")
    return ax


ax = plot_double_differential(E_LO_GEV, E_HI_GEV)
ax.figure.suptitle(dataset_title(data))
ax.figure.tight_layout()
plt.show()

Because the slice is just two numbers, comparing several of them side by side is
the interesting part — the x–y distribution migrates as the energy rises and DIS
takes over from quasi-elastic scattering. Edit `SLICES` to taste; each panel
keeps its own colour scale, so read the colourbars, not the colours.

In [ ]:
SLICES = [(0.5, 1.0), (1.0, 2.0), (2.0, 5.0)]

fig, axes = plt.subplots(1, len(SLICES), figsize=(6 * len(SLICES), 5))
for ax, (lo, hi) in zip(np.atleast_1d(axes), SLICES):
    plot_double_differential(lo, hi, ax=ax)
fig.suptitle(dataset_title(data))
fig.tight_layout()
plt.show()

### Optional: drag the slice around

If `ipywidgets` is installed (it comes with the `notebook` extra), the cell
below gives you two sliders over the energy range actually present in the file.
Without it the cell prints a note and the static cells above remain the way to
change the slice.

In [ ]:
try:
    from ipywidgets import FloatLogSlider, interact
except ImportError:
    print("ipywidgets is not installed — edit E_LO_GEV / E_HI_GEV above instead.")
    print('Install with: pip install -e ".[notebook]"')
else:
    e_min = float(np.min(cols["energy_gev"]))
    e_max = float(np.max(cols["energy_gev"]))

    def _slider(value):
        return FloatLogSlider(
            value=value, base=10,
            min=np.log10(e_min), max=np.log10(e_max),
            step=0.02, readout_format=".2f",
        )

    @interact(e_lo=_slider(max(e_min, E_LO_GEV)), e_hi=_slider(min(e_max, E_HI_GEV)))
    def _show(e_lo, e_hi):
        if e_hi <= e_lo:
            print("upper edge must exceed the lower edge")
            return
        plot_double_differential(e_lo, e_hi)
        plt.show()

## 9. Comparing generators

One file is one job: one generator version on one initial state. The comparison
that motivates the framework needs several, and `figure_channel_comparison`
draws it — one panel per interaction channel, one line per dataset, for a chosen
weak current.

The one rule: **only compare datasets that share an initial state.** Cross
sections on different nuclei, or for different neutrino flavours, are not the
same quantity and do not belong on the same axes. The cell below therefore
groups the discovered files by `(probe, target)` and compares within a group.

The batch equivalent — every comparison a run configuration implies, written to
PNG — is `neutrino-factory plot-output --config <your config>`.

In [ ]:
from neutrino_factory.plots import currents_present, figure_channel_comparison

# Everything we found in section 0; narrow this list to compare specific files.
COMPARE = list(available)

groups: dict[tuple[str, str], list] = {}
for path in COMPARE:
    other = read_plot_data(path)
    groups.setdefault((other.probe, other.target), []).append(other)

for (probe, target), datasets in groups.items():
    if len(datasets) < 2:
        print(f"only one dataset for {probe} on {target} — nothing to compare")
        continue
    currents = {c for other in datasets for c in currents_present(other.is_cc)}
    for current in sorted(currents):
        figure_channel_comparison(
            datasets, current, plt, bins=60, group_label=f"{probe} on {target}"
        )
        plt.show()

## Where to go next

* **`neutrino-factory analyze-kinematics --input <file>`** prints the numbers
  behind section 7: counts per channel, per-variable weighted mean/median/range,
  and the **Kish effective sample size** `n_eff = (Σw)²/Σw²` per channel. Read
  that before trusting any distribution above — GiBUU's quasi-elastic channel
  can show `n_eff/n` below 1 %, meaning a million events carry the statistical
  power of a few thousand.
* **`neutrino-factory plot-output --config <config>`** renders the same plots as
  PNGs for every output of a run, including the cross-generator comparisons.
* **`docs/physics.md`** is the full contract: units and frames, the
  `xsec_weight` definition, the interaction taxonomy, and how chunk merging
  averages rather than sums.
* **`docs/generators/*.md`** explain what each backend actually does and which
  of its quirks the normalizer had to absorb.